In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/loan_features.csv')

X = df.drop(columns=['status'])
y = df['status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)

(118936, 25) (29734, 25)


In [2]:
# Numeric → train median
numeric_to_impute = ['property_value', 'income', 'loan_to_value_ratio',
                     'debt_to_income_ratio', 'term']

for col in numeric_to_impute:
    median_val = X_train[col].median()      # learn from TRAIN only
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col]  = X_test[col].fillna(median_val)   # apply SAME value to test

# Categorical → train mode
categorical_to_impute = ['loan_limit', 'approved_in_advance', 'loan_purpose',
                         'negative_amortization', 'age', 'submission_of_application']

for col in categorical_to_impute:
    mode_val = X_train[col].mode()[0]        # learn from TRAIN only
    X_train[col] = X_train[col].fillna(mode_val)
    X_test[col]  = X_test[col].fillna(mode_val)

# Verify zero NaNs remain in both
print("Train NaNs:", X_train.isnull().sum().sum())
print("Test NaNs:", X_test.isnull().sum().sum())

Train NaNs: 0
Test NaNs: 0


In [3]:
# One-hot encode all categorical (object) columns
X_train_enc = pd.get_dummies(X_train, drop_first=True)
X_test_enc  = pd.get_dummies(X_test, drop_first=True)

# Align columns — test must have identical columns to train
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print(X_train_enc.shape, X_test_enc.shape)

(118936, 40) (29734, 40)


In [4]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)   # fit LEARNS from train, then transforms it
X_test_scaled  = scaler.transform(X_test_enc)         # transform only APPLIES train's numbers

# Save the scaler — you'll need the exact same one in later notebooks
joblib.dump(scaler, '../models/scaler.pkl')
print("Scaled. Train:", X_train_scaled.shape, "Test:", X_test_scaled.shape)

Scaled. Train: (118936, 40) Test: (29734, 40)


In [5]:
print(f"Train mean (should be ~0): {X_train_scaled.mean():.4f}")
print(f"Train std (should be ~1): {X_train_scaled.std():.4f}")
print(f"Test mean (should be ~0): {X_test_scaled.mean():.4f}")
print(f"Test std (should be ~1): {X_test_scaled.std():.4f}")

Train mean (should be ~0): -0.0000
Train std (should be ~1): 1.0000
Test mean (should be ~0): -0.0004
Test std (should be ~1): 0.9876


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, average_precision_score)

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)

y_pred  = logreg.predict(X_test_scaled)
y_proba = logreg.predict_proba(X_test_scaled)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("PR-AUC: ", round(average_precision_score(y_test, y_proba), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-default', 'Default']))

ROC-AUC: 0.8425
PR-AUC:  0.7668

Confusion Matrix:
[[22165   241]
 [ 3721  3607]]

Classification Report:
              precision    recall  f1-score   support

 Non-default       0.86      0.99      0.92     22406
     Default       0.94      0.49      0.65      7328

    accuracy                           0.87     29734
   macro avg       0.90      0.74      0.78     29734
weighted avg       0.88      0.87      0.85     29734



In [7]:
logreg_balanced = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',   # the only change from Cell 4
    random_state=42
)
logreg_balanced.fit(X_train_scaled, y_train)

y_pred_bal  = logreg_balanced.predict(X_test_scaled)
y_proba_bal = logreg_balanced.predict_proba(X_test_scaled)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_bal), 4))
print("PR-AUC: ", round(average_precision_score(y_test, y_proba_bal), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_bal))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_bal, target_names=['Non-default', 'Default']))

ROC-AUC: 0.8437
PR-AUC:  0.7681

Confusion Matrix:
[[19854  2552]
 [ 2511  4817]]

Classification Report:
              precision    recall  f1-score   support

 Non-default       0.89      0.89      0.89     22406
     Default       0.65      0.66      0.66      7328

    accuracy                           0.83     29734
   macro avg       0.77      0.77      0.77     29734
weighted avg       0.83      0.83      0.83     29734



In [8]:
import joblib
import numpy as np

# Scaled features back to DataFrames so column names survive (needed for feature importance in 05)
import pandas as pd
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_enc.columns, index=X_train_enc.index)
X_test_scaled_df  = pd.DataFrame(X_test_scaled,  columns=X_test_enc.columns,  index=X_test_enc.index)

# Save scaled feature sets
X_train_scaled_df.to_csv('../data/X_train_scaled.csv', index=False)
X_test_scaled_df.to_csv('../data/X_test_scaled.csv', index=False)

# Save labels
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

# Save the validated model (and scaler was already saved)
joblib.dump(logreg_balanced, '../models/logreg_balanced.pkl')

print("Saved: X_train_scaled, X_test_scaled, y_train, y_test, logreg_balanced.pkl")

Saved: X_train_scaled, X_test_scaled, y_train, y_test, logreg_balanced.pkl
